In [1]:
import pandas as pd
from pathlib import Path

base_dir = Path.cwd()
SAMPLESHEETS_PATH = base_dir / "preprocessing/mlomix_samplesheets"
if not SAMPLESHEETS_PATH.exists():
    SAMPLESHEETS_PATH = base_dir.parent / "preprocessing/mlomix_samplesheets"

# Set output dir to current dir plus output
OUTPUT_DIR = base_dir / "output"
if not OUTPUT_DIR.exists():
    OUTPUT_DIR.mkdir()

# samplesheet all
SAMPLESHEET_ALL = SAMPLESHEETS_PATH / "samplesheet_all.csv"

# get all training samplesheets
SAMPLESHEET_TRAIN_BIMODAL = SAMPLESHEETS_PATH / "samplesheet_train_bimodal.csv"
SAMPLESHEET_TRAIN_GEX = SAMPLESHEETS_PATH / "samplesheet_train_gex.csv"
SAMPLESHEET_TRAIN_DNAM = SAMPLESHEETS_PATH / "samplesheet_train_dnam.csv"

# get all calibrate samplesheets
SAMPLESHEET_CALIBRATE_GEX = SAMPLESHEETS_PATH / "samplesheet_calibrate_gex.csv"
SAMPLESHEET_CALIBRATE_DNAM = SAMPLESHEETS_PATH / "samplesheet_calibrate_dnam.csv"

# get all test samplesheets
SAMPLESHEET_TEST_BIMODAL = SAMPLESHEETS_PATH / "samplesheet_test_bimodal.csv"
SAMPLESHEET_TEST_GEX = SAMPLESHEETS_PATH / "samplesheet_test_gex.csv"
SAMPLESHEET_TEST_DNAM = SAMPLESHEETS_PATH / "samplesheet_test_dnam.csv"

In [2]:
# Read in samplesheet_all
samplesheet_all = pd.read_csv(SAMPLESHEET_ALL, index_col=0)
print(samplesheet_all.head())

                 data_silo    dataset batch               class  \
sample                                                            
16-001  labeled_diagnostic  Tran-2022   NaN  B-ALL, ETV6::RUNX1   
16-002  labeled_diagnostic  Tran-2022   NaN  B-ALL, ETV6::RUNX1   
16-003  labeled_diagnostic  Tran-2022   NaN  B-ALL, ETV6::RUNX1   
16-006  labeled_diagnostic  Tran-2022   NaN  B-ALL, ETV6::RUNX1   
16-008  labeled_diagnostic  Tran-2022   NaN               T-ALL   

                                  gex_feature_counts_file  \
sample                                                      
16-001  /home/mariya/Data/pallas/raw/Tran-2022/GSE1811...   
16-002  /home/mariya/Data/pallas/raw/Tran-2022/GSE1811...   
16-003  /home/mariya/Data/pallas/raw/Tran-2022/GSE1811...   
16-006  /home/mariya/Data/pallas/raw/Tran-2022/GSE1811...   
16-008  /home/mariya/Data/pallas/raw/Tran-2022/GSE1811...   

       dnam_beta_matrix_file dnam_pvals_file  sentrix_id sentrix_position  \
sample                   

In [3]:
# Read in the other samplesheets
samplesheet_train_bimodal = pd.read_csv(SAMPLESHEET_TRAIN_BIMODAL, index_col=0)
samplesheet_train_gex = pd.read_csv(SAMPLESHEET_TRAIN_GEX, index_col=0)
samplesheet_train_dnam = pd.read_csv(SAMPLESHEET_TRAIN_DNAM, index_col=0)
samplesheet_calibrate_gex = pd.read_csv(SAMPLESHEET_CALIBRATE_GEX, index_col=0)
samplesheet_calibrate_dnam = pd.read_csv(SAMPLESHEET_CALIBRATE_DNAM, index_col=0)
samplesheet_test_bimodal = pd.read_csv(SAMPLESHEET_TEST_BIMODAL, index_col=0)
samplesheet_test_gex = pd.read_csv(SAMPLESHEET_TEST_GEX, index_col=0)
samplesheet_test_dnam = pd.read_csv(SAMPLESHEET_TEST_DNAM, index_col=0)

In [5]:
# Create dataframe with the following columns: sample, dataset, class, batch, data silo, gex data split, dnam data split, bimodal data split
data_report = samplesheet_all[["dataset", "class", "batch", "data_silo"]].copy()

# For each item in data_report, determine the remaining columns as follows:
# - gex data split: if the sample is in samplesheet_train_gex, then "train", if in samplesheet_calibrate_gex, then "calibrate", if in samplesheet_test_gex, then "test", else "none"
data_report["gex_data_split"] = None
data_report.loc[data_report.index.isin(samplesheet_train_gex.index), "gex_data_split"] = "train"
data_report.loc[data_report.index.isin(samplesheet_calibrate_gex.index), "gex_data_split"] = "calibrate"
data_report.loc[data_report.index.isin(samplesheet_test_gex.index), "gex_data_split"] = "test"
# Do the same for dnam
data_report["dnam_data_split"] = None
data_report.loc[data_report.index.isin(samplesheet_train_dnam.index), "dnam_data_split"] = "train"
data_report.loc[data_report.index.isin(samplesheet_calibrate_dnam.index), "dnam_data_split"] = "calibrate"
data_report.loc[data_report.index.isin(samplesheet_test_dnam.index), "dnam_data_split"] = "test"
# Do the same for bimodal
data_report["bimodal_data_split"] = None
data_report.loc[data_report.index.isin(samplesheet_train_bimodal.index), "bimodal_data_split"] = "train"
data_report.loc[data_report.index.isin(samplesheet_test_bimodal.index), "bimodal_data_split"] = "test"

# Sort data
data_report["data_silo"] = pd.Categorical(data_report["data_silo"], categories=["labeled_diagnostic", "unlabeled", "multiclass", "relapse", "replicate", "excluded", "unrecognized"], ordered=True)
data_report = data_report.sort_values(by=["data_silo", "class", "dataset", "batch"])

# Order cols as follows: dataset, class, data silo, gex data split, dnam data split, bimodal data split, batch
data_report = data_report[["dataset", "class", "data_silo", "gex_data_split", "dnam_data_split", "bimodal_data_split", "batch"]]

# Rename index to "Sample name"
data_report.index.name = "Sample name"

# Rename dataset to "Dataset name"
data_report = data_report.rename(columns={
    "dataset": "Dataset name",
    "class": "Subtype",
    "data_silo": "Data silo",
    "gex_data_split": "GEX data split",
    "dnam_data_split": "DNAm data split",
    "bimodal_data_split": "Bimodal data split",
    "batch": "RNA-seq batch"
    })

print(data_report.head())
data_report.to_csv(OUTPUT_DIR / "sample_report.csv")


               Dataset name             Subtype           Data silo  \
Sample name                                                           
AML_046          Krali-2021  AML, APL-PML::RARA  labeled_diagnostic   
AML_059          Krali-2021  AML, APL-PML::RARA  labeled_diagnostic   
AML_075          Krali-2021  AML, APL-PML::RARA  labeled_diagnostic   
AML_106          Krali-2021  AML, APL-PML::RARA  labeled_diagnostic   
SJAML030233_D1  SJC-DS-1009  AML, APL-PML::RARA  labeled_diagnostic   

               GEX data split DNAm data split Bimodal data split RNA-seq batch  
Sample name                                                                     
AML_046                  None           train               None           NaN  
AML_059                  None       calibrate               None           NaN  
AML_075                  None       calibrate               None           NaN  
AML_106                  None       calibrate               None           NaN  
SJAML030233_D1  